In [ ]:
!nvidia-smi


In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base = "/content/drive/MyDrive/RobotexNationals"

folders = [
    "datasets",
    "models",
    "runs",
    "exports",
    "notebooks"
]

for folder in folders:
    os.makedirs(os.path.join(base, folder), exist_ok=True)

print("Folders created successfully!")

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/RobotexNationals/datasets/Candle.yolov8.zip"
extract_path = "/content/dataset"
integrity_check_output_file = "/tmp/unzip_integrity_check_output.txt"

print(f"Investigating the zip file: {zip_path}")

# First, let's explicitly check the file type using `file` command
print(f"Checking file type of '{zip_path}':")
!file "{zip_path}"

# Perform an integrity check using `unzip -t` and save output to a file
print(f"Performing integrity check on '{zip_path}' using `unzip -t` and saving output to {integrity_check_output_file}...")
!unzip -t "{zip_path}" > "{integrity_check_output_file}" 2>&1

# Display the full integrity check output from the file
print(f"\nFull Integrity Check Output from {integrity_check_output_file}:")
with open(integrity_check_output_file, 'r') as f:
    print(f.read())

# Now, attempting the extraction with Python's zipfile again, with error handling.
print(f"Attempting extraction of '{zip_path}' to '{extract_path}' with Python's zipfile module...")
extraction_successful = False
try:
    # Ensure the extract directory exists
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Dataset extracted successfully with Python's zipfile!")
    extraction_successful = True
except zipfile.BadZipFile:
    print("\n--- DIAGNOSIS ---")
    print(f"Error: Python's `zipfile` module reported 'BadZipFile' during extraction.")
    print(f"Despite `unzip -t` possibly showing 'OK' for individual files, this indicates the zip's central directory or overall structure is still problematic for Python's library.")
    print(f"Action Required: Please obtain a new, valid copy of the `Candle.yolov8.zip` file, or upload the unzipped dataset directly to your Google Drive.")
    print("-----------------")
except Exception as e:
    print(f"An unexpected error occurred during extraction with Python's zipfile: {type(e).__name__}: {e}")


# List contents of the extracted directory for verification
print(f"\nContents of {extract_path}:")
if os.path.exists(extract_path) and os.listdir(extract_path):
    for root, dirs, files in os.walk(extract_path):
        level = root.replace(extract_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')
else:
    if not extraction_successful:
        print("Extraction directory is empty or does not exist because extraction failed.")
    else:
        print("Extraction directory is empty or does not exist (unexpected for successful extraction).")

In [ ]:
data_yaml_path = os.path.join(extract_path, 'data.yaml')

print(f"Checking if data.yaml exists at: {data_yaml_path}")
if os.path.exists(data_yaml_path):
    print(f"Success! The data.yaml file is located at: {data_yaml_path}")
else:
    print(f"Error: data.yaml not found at {data_yaml_path}")

print(f"\nFull recursive listing of '{extract_path}':")
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0
)

In [ ]:
!pip install ultralytics opencv-python

In [ ]:
import os

video_folder = "/content/drive/MyDrive/RobotexNationals/videos"

videos = [
    f for f in os.listdir(video_folder)
    if f.endswith((".mp4", ".avi", ".mov"))
]

print(videos)

In [ ]:
!pip install ultralytics opencv-python -q

from ultralytics import YOLO
import cv2
import os

# Load your trained model
model = YOLO("/content/drive/MyDrive/RobotexNationals/models/best.pt")

In [ ]:
import os
import cv2

output_root = "/content/drive/MyDrive/RobotexNationals/extracted_frames_raw"

os.makedirs(output_root, exist_ok=True)

for video in videos:

    print("="*60)
    print(f"Processing {video}")

    video_path = os.path.join(video_folder, video)

    video_name = os.path.splitext(video)[0]

    output_folder = os.path.join(output_root, video_name)

    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Could not open:", video)
        continue

    fps = int(cap.get(cv2.CAP_PROP_FPS))

    frame_interval = max(1, fps // 2)   # One frame every 0.5 second

    frame_number = 0
    saved = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_number % frame_interval == 0:

            filename = os.path.join(
                output_folder,
                f"{video_name}_{saved:05d}.jpg"
            )

            cv2.imwrite(filename, frame)

            saved += 1

        frame_number += 1

    cap.release()

    print(f"Saved {saved} frames.")

In [ ]:
import os

frames_root = "/content/drive/MyDrive/RobotexNationals/extracted_frames_raw"

for folder in sorted(os.listdir(frames_root)):
    path = os.path.join(frames_root, folder)

    if os.path.isdir(path):
        print(folder, len(os.listdir(path)))

In [ ]:
import glob

for folder in sorted(os.listdir(frames_root)):
    folder_path = os.path.join(frames_root, folder)

    if os.path.isdir(folder_path):
        images = glob.glob(os.path.join(folder_path, "*.jpg"))
        print(f"{folder}: {len(images)} JPG images")

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/RobotexNationals/models/best.pt")

In [ ]:
import os
from glob import glob

frames_root = "/content/drive/MyDrive/RobotexNationals/extracted_frames_raw"

project_path = "/content/drive/MyDrive/RobotexNationals/auto_labels"

os.makedirs(project_path, exist_ok=True)

folders = sorted(os.listdir(frames_root))

for folder in folders:

    folder_path = os.path.join(frames_root, folder)

    # Skip if not a directory
    if not os.path.isdir(folder_path):
        continue

    images = glob(os.path.join(folder_path, "*.jpg"))

    # Skip empty folders
    if len(images) == 0:
        print(f"Skipping {folder} (No images)")
        continue

    print("="*60)
    print(f"Auto-labeling: {folder}")
    print(f"Images: {len(images)}")

    model.predict(
        source=folder_path,
        save=False,
        save_txt=True,
        save_conf=False,
        conf=0.15,
        project=project_path,
        name=folder,
        exist_ok=True,
        verbose=False
    )

print("\n🎉 Auto-labeling completed!")

In [ ]:
import os

labels_root = "/content/drive/MyDrive/RobotexNationals/auto_labels"

for folder in sorted(os.listdir(labels_root)):

    label_folder = os.path.join(labels_root, folder, "labels")

    if os.path.exists(label_folder):

        txt_files = [f for f in os.listdir(label_folder) if f.endswith(".txt")]

        print(f"{folder}: {len(txt_files)} labels")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

# ==========================
# PATHS
# ==========================

zip_path = "/content/drive/MyDrive/RobotexNationals/datasets/robotex.v1i.yolov11.zip"

extract_path = "/content/drive/MyDrive/RobotexNationals/datasets/robotex_v2"

integrity_output = "/tmp/unzip_check.txt"

# ==========================
# CHECK FILE
# ==========================

print("="*60)
print("Checking ZIP file...")
print("="*60)

!file "{zip_path}"

print("\nFile Size:")
print(f"{os.path.getsize(zip_path)/1024/1024:.2f} MB")

# ==========================
# CHECK ZIP INTEGRITY
# ==========================

print("\nRunning ZIP integrity check...")

!unzip -t "{zip_path}" > "{integrity_output}" 2>&1

with open(integrity_output,"r") as f:
    print(f.read())

# ==========================
# EXTRACT
# ==========================

print("\nExtracting dataset...")

os.makedirs(extract_path, exist_ok=True)

try:

    with zipfile.ZipFile(zip_path,'r') as zip_ref:

        print("\nFirst few files inside ZIP:")

        print(zip_ref.namelist()[:15])

        zip_ref.extractall(extract_path)

    print("\n✅ Extraction Successful!")

except zipfile.BadZipFile:

    print("❌ Bad Zip File!")

except Exception as e:

    print(type(e).__name__, e)

# ==========================
# VERIFY DATASET
# ==========================

print("\n")
print("="*60)
print("Verifying Dataset Structure")
print("="*60)

for folder in ["train","valid","test"]:

    image_path=os.path.join(extract_path,folder,"images")

    label_path=os.path.join(extract_path,folder,"labels")

    if os.path.exists(image_path):

        print(f"\n{folder.upper()}")

        print("Images :",len(os.listdir(image_path)))

        print("Labels :",len(os.listdir(label_path)))

print("\nDone.")

In [ ]:
import os

zip_path="/content/drive/MyDrive/RobotexNationals/datasets/robotex.v1i.yolov11.zip"

print(os.path.exists(zip_path))
print(os.path.getsize(zip_path))

In [ ]:
import os

dataset_folder = "/content/drive/MyDrive/RobotexNationals/datasets"

print(os.listdir(dataset_folder))

In [ ]:
from google.colab import drive

drive.flush_and_unmount()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

dataset_folder = "/content/drive/MyDrive/RobotexNationals/datasets"

print("Files inside datasets folder:\n")

for file in os.listdir(dataset_folder):
    print(file)

In [ ]:
import os

dataset_folder = "/content/drive/MyDrive/RobotexNationals/datasets"

for file in os.listdir(dataset_folder):
    path = os.path.join(dataset_folder, file)

    if os.path.isfile(path):
        print(f"{file}")
        print(f"Size : {os.path.getsize(path)/1024/1024:.2f} MB")
        print("-"*40)

In [ ]:
import os

zip_path="/content/drive/MyDrive/RobotexNationals/datasets/robotex.v1i.yolov11.zip"

print(os.path.exists(zip_path))

print(os.path.getsize(zip_path)/1024/1024)

In [ ]:
!pip install -q roboflow

from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY_HERE")

project = rf.workspace("nidhi-thakur-nvlrf").project("robotex-ph10h")

version = project.version(1)

dataset = version.download("yolov11")

In [ ]:
print(dataset.location)

In [ ]:
import os

dataset_path = "/content/robotex-1"

for split in ["train", "valid", "test"]:

    img = os.path.join(dataset_path, split, "images")
    lbl = os.path.join(dataset_path, split, "labels")

    print("="*50)
    print(split.upper())

    print("Images :", len(os.listdir(img)))
    print("Labels :", len(os.listdir(lbl)))

In [ ]:
with open("/content/robotex-1/data.yaml", "r") as f:
    print(f.read())

In [ ]:
yaml_content = """
path: /content/robotex-1

train: train/images
val: valid/images
test: test/images

nc: 2

names:
  0: candle
  1: wall
"""

with open("/content/robotex-1/data.yaml", "w") as f:
    f.write(yaml_content)

print("✅ data.yaml updated")

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/RobotexNationals"):
    for file in files:
        if file.endswith(".pt"):
            print(os.path.join(root, file))

In [ ]:
from ultralytics import YOLO

model = YOLO("//content/drive/MyDrive/RobotexNationals/models/best.pt")

results = model.train(

    data="/content/robotex-1/data.yaml",

    epochs=40,

    imgsz=640,

    batch=16,

    workers=2,

    optimizer="AdamW",

    lr0=0.0005,

    cos_lr=True,

    patience=10,

    cache=True,

    mosaic=1.0,

    mixup=0.1,

    degrees=8,

    translate=0.08,

    scale=0.30,

    fliplr=0.0,

    flipud=0.0,

    hsv_h=0.015,

    hsv_s=0.5,

    hsv_v=0.3,

    project="/content/drive/MyDrive/RobotexNationals/training",

    name="robotex_v2",

    exist_ok=True
)

In [ ]:
import os

path = "/content/drive/MyDrive/RobotexNationals"

print(os.listdir(path))

In [ ]:
video_path = "/content/drive/MyDrive/RobotexNationals"

for item in os.listdir(video_path):
    print(item)

In [ ]:
from google.colab import drive

drive.flush_and_unmount()

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

root = "/content/drive/MyDrive/RobotexNationals"

print("Contents of RobotexNationals:\n")

for item in sorted(os.listdir(root)):
    print(item)

In [ ]:
import os

video_folder = "/content/drive/MyDrive/RobotexNationals/webcam_videos"

print("Folder exists:", os.path.exists(video_folder))

if os.path.exists(video_folder):

    videos = sorted([
        f for f in os.listdir(video_folder)
        if f.lower().endswith((".mp4", ".avi", ".mov"))
    ])

    print(f"\nFound {len(videos)} videos:\n")

    for video in videos:
        print(video)

In [ ]:
import os

video_folder = "/content/drive/MyDrive/RobotexNationals/webcam_videos"

print("Video Sizes:\n")

for file in sorted(os.listdir(video_folder)):

    path = os.path.join(video_folder, file)

    if os.path.isfile(path):

        size = os.path.getsize(path) / (1024 * 1024)

        print(f"{file:<20} {size:.2f} MB")

In [ ]:
import cv2
import os

video_folder = "/content/drive/MyDrive/RobotexNationals/webcam_videos"

output_root = "/content/drive/MyDrive/RobotexNationals/webcam_frames"

os.makedirs(output_root, exist_ok=True)

FRAME_INTERVAL = 8

videos = sorted([
    f for f in os.listdir(video_folder)
    if f.lower().endswith((".mp4", ".avi", ".mov"))
])

print(f"Found {len(videos)} videos\n")

for video in videos:

    print("=" * 60)
    print(f"Processing {video}")

    video_path = os.path.join(video_folder, video)

    folder_name = os.path.splitext(video)[0]

    output_folder = os.path.join(output_root, folder_name)

    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    frame_no = 0
    saved = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_no % FRAME_INTERVAL == 0:

            filename = os.path.join(
                output_folder,
                f"{folder_name}_{saved:05d}.jpg"
            )

            cv2.imwrite(filename, frame)

            saved += 1

        frame_no += 1

    cap.release()

    print(f"Saved {saved} frames")

In [ ]:
from ultralytics import YOLO

model = YOLO(
"/content/drive/MyDrive/RobotexNationals/models/robotex.pt"
)

print("Model Loaded Successfully ✅")

In [ ]:
import os
from glob import glob

frames_root = "/content/drive/MyDrive/RobotexNationals/webcam_frames"

project_root = "/content/drive/MyDrive/RobotexNationals/webcam_auto_labels"

os.makedirs(project_root, exist_ok=True)

folders = sorted(os.listdir(frames_root))

for folder in folders:

    # Skip hidden folders like .ipynb_checkpoints
    if folder.startswith("."):
        continue

    folder_path = os.path.join(frames_root, folder)

    if not os.path.isdir(folder_path):
        continue

    images = glob(os.path.join(folder_path, "*.jpg"))

    if len(images) == 0:
        print(f"Skipping {folder} (No images)")
        continue

    print("="*60)
    print(f"Auto-labeling {folder}")
    print(f"Images: {len(images)}")

    model.predict(
        source=folder_path,
        save=False,
        save_txt=True,
        save_conf=False,
        conf=0.25,
        imgsz=640,
        project=project_root,
        name=folder,
        exist_ok=True,
        verbose=False
    )

print("\n✅ Auto-labeling Completed!")

In [ ]:
import os

root = "/content/drive/MyDrive/RobotexNationals/webcam_auto_labels"

total = 0

for folder in sorted(os.listdir(root)):

    if folder.startswith("."):
        continue

    label_folder = os.path.join(root, folder, "labels")

    if os.path.exists(label_folder):

        count = len([
            f for f in os.listdir(label_folder)
            if f.endswith(".txt")
        ])

        total += count

        print(f"{folder}: {count}")

print("\nTotal Labels:", total)

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY_HERE")
project = rf.workspace("nidhi-thakur-nvlrf").project("robotex-final")
version = project.version(1)
dataset = version.download("yolov11")


In [ ]:
print(dataset.location)

In [ ]:
import os

dataset_path = dataset.location

for split in ["train","valid","test"]:

    img = os.path.join(dataset_path, split, "images")
    lbl = os.path.join(dataset_path, split, "labels")

    print("="*50)
    print(split.upper())

    print("Images :", len(os.listdir(img)))
    print("Labels :", len(os.listdir(lbl)))

In [ ]:
yaml = f"""
path: {dataset.location}

train: train/images
val: valid/images
test: test/images

nc: 2

names:
  0: candle
  1: wall
"""

with open(os.path.join(dataset.location, "data.yaml"), "w") as f:
    f.write(yaml)

print("✅ data.yaml updated")

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/RobotexNationals/models/robotex.pt")

In [ ]:
results = model.train(

    data=os.path.join(dataset.location, "data.yaml"),

    epochs=30,

    imgsz=640,

    batch=16,

    workers=2,

    optimizer="AdamW",

    lr0=0.0003,

    cos_lr=True,

    patience=8,

    cache=True,

    project="/content/drive/MyDrive/RobotexNationals/final_training",

    name="competition_final",

    exist_ok=True,

    pretrained=True,

    augment=True
)

In [ ]:
import os
import cv2

# -----------------------------
# PATHS
# -----------------------------

video_folder = "/content/drive/MyDrive/RobotexNationals/webcam_closeups"

output_root = "/content/drive/MyDrive/RobotexNationals/webcam_closeups"

os.makedirs(output_root, exist_ok=True)

# Save every 6th frame
FRAME_INTERVAL = 12

videos = sorted([
    f for f in os.listdir(video_folder)
    if f.lower().endswith((".mp4", ".avi", ".mov"))
])

print(f"Found {len(videos)} videos")

for v in videos:
    print(v)

In [ ]:
import cv2
import os

total_saved = 0

for video in videos:

    print("="*60)
    print(f"Processing {video}")

    video_path = os.path.join(video_folder, video)

    folder_name = os.path.splitext(video)[0]

    output_folder = os.path.join(output_root, folder_name)

    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    frame_no = 0
    saved = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_no % FRAME_INTERVAL == 0:

            filename = os.path.join(
                output_folder,
                f"{folder_name}_{saved:05d}.jpg"
            )

            cv2.imwrite(filename, frame)

            saved += 1

        frame_no += 1

    cap.release()

    total_saved += saved

    print(f"Saved {saved} frames")

print("\n" + "="*60)
print(f"TOTAL IMAGES SAVED : {total_saved}")
print("="*60)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/RobotexNationals/final_training/competition_final/weights/competition_final.pt")

In [ ]:
import os

frames_root = "/content/drive/MyDrive/RobotexNationals/webcam_closeups"

output_root = "/content/drive/MyDrive/RobotexNationals/webcam_auto_labels_v2"

os.makedirs(output_root, exist_ok=True)

In [ ]:
import os

folders = sorted([
    f for f in os.listdir(frames_root)
    if os.path.isdir(os.path.join(frames_root, f))
])

print(f"Found {len(folders)} folders")

for folder in folders:

    print("="*60)
    print("Auto-labeling:", folder)

    model.predict(

        source=os.path.join(frames_root, folder),

        save=False,

        save_txt=True,

        save_conf=False,

        conf=0.10,          # Lower threshold to capture more detections

        project=output_root,

        name=folder,

        exist_ok=True,

        verbose=False

    )

    labels_folder = os.path.join(output_root, folder, "labels")

    if os.path.exists(labels_folder):

        n = len([
            f for f in os.listdir(labels_folder)
            if f.endswith(".txt")
        ])

        print(f"Labels Created : {n}")

    else:

        print("No labels created.")

In [ ]:
import os

print("="*60)

total = 0

for folder in folders:

    labels_folder = os.path.join(output_root, folder, "labels")

    if os.path.exists(labels_folder):

        count = len([
            f for f in os.listdir(labels_folder)
            if f.endswith(".txt")
        ])

        total += count

        print(folder, ":", count)

print("="*60)
print("TOTAL LABEL FILES :", total)

In [ ]:
import os

missing = []

for folder in folders:

    image_folder = os.path.join(frames_root, folder)
    label_folder = os.path.join(output_root, folder, "labels")

    imgs = set(
        f[:-4] for f in os.listdir(image_folder)
        if f.endswith(".jpg")
    )

    lbls = set(
        f[:-4] for f in os.listdir(label_folder)
        if f.endswith(".txt")
    )

    diff = imgs - lbls

    if len(diff) > 0:

        print(f"{folder}: {len(diff)} missing labels")

        missing.extend([
            os.path.join(image_folder, f + ".jpg")
            for f in diff
        ])

print("\nTotal Missing:", len(missing))

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

for img_path in missing[:20]:   # show first 20

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.title(os.path.basename(img_path))
    plt.axis("off")
    plt.show()

In [ ]:
import os

root = "/content/drive/MyDrive/RobotexNationals"

print("Searching for .pt files...\n")

for folder, subfolders, files in os.walk(root):
    for file in files:
        if file.endswith(".pt"):
            print(os.path.join(folder, file))

In [ ]:
from ultralytics import YOLO

models = {
    "best": "/content/drive/MyDrive/RobotexNationals/models/best.pt",
    "robotex": "/content/drive/MyDrive/RobotexNationals/models/robotex.pt",
    "robotex_v2_best": "/content/drive/MyDrive/RobotexNationals/training/robotex_v2/weights/best.pt"
}

for name, path in models.items():

    print("="*60)
    print(name)

    model = YOLO(path)

    print(model.names)

In [ ]:
from ultralytics import YOLO

image = "/content/drive/MyDrive/RobotexNationals/webcam_closeups/v6/v6_00012.jpg"

models = {
    "best": "/content/drive/MyDrive/RobotexNationals/models/best.pt",
    "robotex": "/content/drive/MyDrive/RobotexNationals/models/robotex.pt",
    "robotex_v2": "/content/drive/MyDrive/RobotexNationals/training/robotex_v2/weights/best.pt"
}

for name, path in models.items():

    print("="*60)
    print(name)

    model = YOLO(path)

    model.predict(
        source=image,
        conf=0.25,
        save=True,
        project="/content/compare_models",
        name=name,
        exist_ok=True
    )

In [ ]:
import cv2
import os

# =====================================================
# PATHS
# =====================================================

video_folder = "/content/drive/MyDrive/RobotexNationals/webcam_closeup_videos"

output_root = "/content/drive/MyDrive/RobotexNationals/webcam_closeups"

os.makedirs(output_root, exist_ok=True)

FRAME_INTERVAL = 12

videos = sorted([
    f for f in os.listdir(video_folder)
    if f.lower().endswith((".mp4", ".avi", ".mov"))
])

print(f"Found {len(videos)} videos\n")

for video in videos:
    print(video)

In [ ]:
import cv2
import os

total_saved = 0

for video in videos:

    print("=" * 60)
    print(f"Processing {video}")

    video_path = os.path.join(video_folder, video)

    folder_name = os.path.splitext(video)[0]

    output_folder = os.path.join(output_root, folder_name)

    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    frame_no = 0
    saved = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_no % FRAME_INTERVAL == 0:

            filename = os.path.join(
                output_folder,
                f"{folder_name}_{saved:05d}.jpg"
            )

            cv2.imwrite(filename, frame)

            saved += 1

        frame_no += 1

    cap.release()

    total_saved += saved

    print(f"Saved {saved} frames")

print("\n" + "=" * 60)
print(f"TOTAL IMAGES SAVED : {total_saved}")
print("=" * 60)

In [ ]:
import os

print("=" * 60)

grand_total = 0

for folder in sorted(os.listdir(output_root)):

    folder_path = os.path.join(output_root, folder)

    if os.path.isdir(folder_path):

        count = len([
            f for f in os.listdir(folder_path)
            if f.endswith(".jpg")
        ])

        grand_total += count

        print(f"{folder} : {count}")

print("=" * 60)
print(f"Grand Total : {grand_total}")

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY_HERE")
project = rf.workspace("nidhi-thakur-nvlrf").project("robotexv2_final")
version = project.version(2)
dataset = version.download("yolov11")


In [ ]:
import os

print(dataset.location)

In [ ]:
from ultralytics import YOLO

# Load your previous competition model
model = YOLO("/content/drive/MyDrive/RobotexNationals/final_training/competition_final/weights/competition_final.pt")

# Fine-tune
model.train(

    data=f"{dataset.location}/data.yaml",

    epochs=15,

    imgsz=640,

    batch=16,

    patience=5,

    project="/content/drive/MyDrive/RobotexNationals/final_training",

    name="competition_final_v2",

    exist_ok=True

)

In [ ]:
import cv2
import os

# ==========================
# PATHS
# ==========================

video_folder = "/content/drive/MyDrive/RobotexNationals/arena_like_videos"

output_root = "/content/drive/MyDrive/RobotexNationals/arena_like_frames"

os.makedirs(output_root, exist_ok=True)

FRAME_INTERVAL = 15

videos = sorted([
    f for f in os.listdir(video_folder)
    if f.lower().endswith((".mp4", ".avi", ".mov"))
])

print(f"Found {len(videos)} videos\n")

total_saved = 0

for video in videos:

    print("="*60)
    print("Processing", video)

    video_path = os.path.join(video_folder, video)

    folder_name = os.path.splitext(video)[0]

    output_folder = os.path.join(output_root, folder_name)

    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    frame_no = 0
    saved = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_no % FRAME_INTERVAL == 0:

            filename = os.path.join(
                output_folder,
                f"{folder_name}_{saved:05d}.jpg"
            )

            cv2.imwrite(filename, frame)

            saved += 1

        frame_no += 1

    cap.release()

    total_saved += saved

    print(f"Saved {saved} frames")

print("\n" + "="*60)
print(f"TOTAL IMAGES SAVED : {total_saved}")
print("="*60)

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY_HERE")
project = rf.workspace("nidhi-thakur-nvlrf").project("final_arena-ojy0j")
version = project.version(1)
dataset = version.download("yolov11")


In [ ]:
print(dataset.location)

In [ ]:
from ultralytics import YOLO

# Continue from your latest best model
model = YOLO(
    "/content/drive/MyDrive/RobotexNationals/final_training/competition_final_v2/weights/robotex_final.pt"
)

model.train(

    data=f"{dataset.location}/data.yaml",

    epochs=10,

    imgsz=640,

    batch=16,

    patience=5,

    workers=2,

    cache=True,

    project="/content/drive/MyDrive/RobotexNationals/final_training",

    name="competition_final_v3",

    exist_ok=True,

    pretrained=True

)